In [1]:
# Load packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle

from FDApy import DenseFunctionalData, MultivariateFunctionalData
from FDApy.representation import DenseArgvals, DenseValues
from FDApy.preprocessing import MFPCA
from FDApy.visualization import plot, plot_multivariate

In [2]:
# Load shots
with open('./data/players_shots_density_attempted.pickle', 'rb') as f:
    players_shots_density = pickle.load(f)
with open('./data/players_shots_density_made.pickle', 'rb') as f:
    players_shots_density_made = pickle.load(f)

In [3]:
# Create functional data
argvals = DenseArgvals({
    'input_dim_0': np.linspace(0, 1, 201),
    'input_dim_1': np.linspace(0, 1, 201)
})

values_shots = DenseValues(
    np.stack([pp / np.max(pp) for pp in players_shots_density['DENSITY'].to_numpy()])
)

values_made = DenseValues(
    np.stack([pp / np.max(pp) for pp in players_shots_density_made['DENSITY'].to_numpy()])
)


fdata_shots = DenseFunctionalData(argvals, values_shots)
fdata_made = DenseFunctionalData(argvals, values_made)
fdata = MultivariateFunctionalData([fdata_shots, fdata_made])

In [4]:
from FDApy.misc.utils import _integrate

def distance_L2(fdata, idx_1, idx_2):
    integral = 0
    for idx in np.arange(fdata.n_functional):
        integral += _integrate(
            (fdata.data[idx].values[idx_1] - fdata.data[idx].values[idx_2])**2,
            fdata.data[idx].argvals['input_dim_0'],
            fdata.data[idx].argvals['input_dim_1']
        )
    return np.sqrt(integral)

In [5]:
# Compute functional distances
distance_matrix = np.zeros((fdata.n_obs, fdata.n_obs))
for i in np.arange(fdata.n_obs):
    for j in np.arange(fdata.n_obs):
        distance_matrix[i, j] = distance_L2(fdata, i, j)

In [6]:
# MFPCA
mfpca = MFPCA(n_components=20, method='inner-product')
mfpca.fit(fdata)

/Users/steven/Documents/workspace/Python/FDApy/FDApy/representation/functional_data.py:1180: UserWarning: The estimation of the variance of the noise is not performed for data with dimension larger than 1 and is set to 0.
  warnings.warn(


In [7]:
# Compute the scores
scores = mfpca.transform(method='InnPro')

In [8]:
# Inverse transform
fdata_recons = mfpca.inverse_transform(scores)

In [9]:
# Save functional distances
with open('./data/distance_functional.pickle', 'wb') as f:
    pickle.dump(distance_matrix, f, protocol=pickle.HIGHEST_PROTOCOL)

In [10]:
# Save MFPCA results
with open('./data/MFPCA.pickle', 'wb') as f:
    pickle.dump(mfpca, f, protocol=pickle.HIGHEST_PROTOCOL)

In [11]:
# Save scores
with open('./data/scores.pickle', 'wb') as f:
    pickle.dump(scores, f, protocol=pickle.HIGHEST_PROTOCOL)

In [12]:
# Save reconstruction
with open('./data/MFPCA_reconstruction.pickle', 'wb') as f:
    pickle.dump(fdata_recons, f, protocol=pickle.HIGHEST_PROTOCOL)